# Correlation-Weighted Edges: Does Expression Correlation Help the GCN-GAE?

**Question tested:** Instead of feeding raw gene expression as node features (tested separately and found to hurt performance), can we use expression **correlation as an edge-confidence weight** on top of the existing per-algorithm graphs -- without changing which edges exist, and without duplicating what correlation-based algorithms (CLR, PCIT, MRNET, ARACNE) already vote on?

**Setup:** Same architecture as `FINAL_MODEL_GCN.ipynb` (3-layer GCN encoder + GAE, identity node features, DREAM4 network 1, 17 algorithms, threshold 0.6). The only difference between the two conditions below is whether each edge carries weight `1` (baseline) or `|expression correlation|` (correlation-weighted) into the GCNConv layers.

**Result (spoiler): no improvement.** AUROC was essentially unchanged (0.8071 -> 0.8070) and AUPR/F1 were slightly worse. Recorded here so this specific idea doesn't get re-tried from scratch later -- see the Conclusion cell at the end for the full discussion.

## 1. Imports and setup

In [ ]:
import time
import random
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn.functional as F
from torch.nn import BatchNorm1d
from torch_geometric.nn import GCNConv, GAE
from torch_geometric.utils import from_scipy_sparse_matrix
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

## 2. Load data

- `normalized_weight_matrix_all_GRN_100_01_0.2filter.csv` -- 17 GRN algorithms' predicted adjacency matrices for DREAM4 in-silico network 1 (100 genes)
- `dream4_100_01_gs.csv` -- ground truth (gold standard) network, 175 known true edges
- Real expression data (knockdowns + knockouts + timeseries + wildtype) -- used here only to build a gene-gene correlation matrix, **not** as node features (that was tested separately and found to hurt performance -- see project notes).

In [ ]:
data = pd.read_csv("/Users/nirwantandukar/input_data/Database.DREAM4/NORMALIZED_EXP/normalized_weight_matrix_all_GRN_100_01_0.2filter.csv", index_col=0)
ground_truth = pd.read_csv("/Users/nirwantandukar/input_data/Database.DREAM4/GS/dream4_100_01_gs.csv", index_col=0)

num_genes = data.shape[0]
gene_names = data.index.tolist()
algorithms = sorted(set(col.split('.')[0] for col in data.columns))
node_features = np.identity(num_genes)  # identity validated as the better choice; unchanged here

print(f"Genes: {num_genes} | Algorithms: {len(algorithms)} | True edges: {int(ground_truth.values.sum())}")

In [ ]:
# Real expression data -- knockdowns, knockouts, timeseries, wildtype -- combined
# into one genes x conditions matrix, used ONLY to compute gene-gene correlation.
base = "/Users/nirwantandukar/input_data/Database.DREAM4_expression/insilico_size100_1/"
kd = pd.read_csv(base + "insilico_size100_1_knockdowns.tsv", sep="\t")
ko = pd.read_csv(base + "insilico_size100_1_knockouts.tsv", sep="\t")
ts = pd.read_csv(base + "insilico_size100_1_timeseries.tsv", sep="\t").drop(columns=["Time"])
wt = pd.read_csv(base + "insilico_size100_1_wildtype.tsv", sep="\t")

gene_order = [f"G{i}" for i in range(1, num_genes + 1)]
combined = pd.concat([kd[gene_order], ko[gene_order], ts[gene_order], wt[gene_order]], axis=0)
expr = combined[gene_order].values.T  # genes x conditions

gene_corr = np.corrcoef(expr)
np.fill_diagonal(gene_corr, 0)
print(f"Correlation matrix shape: {gene_corr.shape}, range [{gene_corr.min():.3f}, {gene_corr.max():.3f}]")

## 3. Two edge-building schemes

Both schemes use the **exact same edges** (same graph topology per algorithm, threshold 0.6) -- the only difference is the weight carried on each edge into `GCNConv`:

- **Baseline:** every edge has implicit weight 1 (GCNConv's default internal normalization)
- **Correlation-weighted:** every edge's weight = `|expression correlation|` between that gene pair, floored at 0.05 so no edge is fully zeroed out

Important implementation note: the Laplacian-normalized weight used to *build* the edge list is **not** the same thing passed to `GCNConv` as `edge_weight` -- that normalized Laplacian contains negative values (it's `I - D^-1/2 A D^-1/2`), and feeding negative weights into `GCNConv`'s own internal degree normalization produces `NaN` (square root of a negative number). The correlation weight passed to `GCNConv` must be a genuine non-negative confidence value.

In [ ]:
def normalize_adj(adj):
    """Normalized Laplacian -- used only to determine which edges exist (topology)."""
    adj = sp.coo_matrix(adj)
    rowsum = np.array(adj.sum(1)).flatten()
    d_inv_sqrt = np.power(rowsum + 1e-5, -0.5)
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
    d_mat_inv_sqrt = sp.diags(d_inv_sqrt)
    identity = sp.eye(adj.shape[0])
    laplacian = identity - d_mat_inv_sqrt.dot(adj).dot(d_mat_inv_sqrt)
    return laplacian.tocoo()


def build_edge_index_baseline(alg_matrix, threshold=0.6):
    """Binary topology, edge_weight=None (GCNConv treats every edge as weight=1)."""
    binary = (alg_matrix >= threshold).astype(float)
    adj_sparse = sp.csr_matrix(binary)
    adj_normalized = normalize_adj(adj_sparse + sp.eye(adj_sparse.shape[0]))
    edge_index, _ = from_scipy_sparse_matrix(adj_normalized)
    return edge_index, None


def build_edge_index_corr_weighted(alg_matrix, corr_matrix, threshold=0.6):
    """Same topology as baseline, but each edge carries |correlation| as its
    confidence weight, floored so no edge goes to zero."""
    binary = (alg_matrix >= threshold).astype(float)
    adj_sparse = sp.csr_matrix(binary)
    adj_normalized = normalize_adj(adj_sparse + sp.eye(adj_sparse.shape[0]))
    edge_index, _ = from_scipy_sparse_matrix(adj_normalized)

    src, dst = edge_index[0].numpy(), edge_index[1].numpy()
    confidence = np.abs(corr_matrix[src, dst])
    confidence = np.clip(confidence, 0.05, 1.0)
    edge_weight = torch.tensor(confidence, dtype=torch.float)
    return edge_index, edge_weight


alg_edges_baseline = {}
alg_edges_weighted = {}
for alg in algorithms:
    cols = [c for c in data.columns if c.startswith(alg + '.')]
    mat = data[cols].values.reshape((num_genes, num_genes))
    alg_edges_baseline[alg] = build_edge_index_baseline(mat, threshold=0.6)
    alg_edges_weighted[alg] = build_edge_index_corr_weighted(mat, gene_corr, threshold=0.6)

## 4. GCN-GAE model (same architecture as FINAL_MODEL_GCN.ipynb, extended to accept edge_weight)

In [ ]:
class GCNEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=3, dropout_rate=0.01):
        super().__init__()
        self.num_layers = num_layers
        self.convs = torch.nn.ModuleList()
        self.convs.append(GCNConv(in_channels, 2 * hidden_channels, cached=False))
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(2 * hidden_channels, 2 * hidden_channels, cached=False))
        self.convs.append(GCNConv(2 * hidden_channels, out_channels, cached=False))
        self.bns = torch.nn.ModuleList([BatchNorm1d(2 * hidden_channels, momentum=0.1) for _ in range(num_layers - 1)])
        self.dropout = torch.nn.Dropout(p=dropout_rate)

    def forward(self, x, edge_index, edge_weight=None):
        x = self.convs[0](x, edge_index, edge_weight)
        x = F.relu(x)
        for i in range(1, self.num_layers - 1):
            x_res = x
            x = self.convs[i](x, edge_index, edge_weight)
            x = self.bns[i - 1](x)
            x = F.relu(x)
            x = self.dropout(x)
            x = x + x_res
        x = self.convs[-1](x, edge_index, edge_weight)
        x = F.relu(x)
        return x


def train_gae(edge_index, edge_weight, x, embedding_dim=16, epochs=150, weight_decay=1e-8):
    encoder = GCNEncoder(in_channels=x.shape[1], hidden_channels=embedding_dim, out_channels=embedding_dim)
    model = GAE(encoder).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=weight_decay)
    model.train()
    for _ in range(epochs):
        opt.zero_grad()
        z = model.encode(x, edge_index, edge_weight)
        loss = model.recon_loss(z, edge_index)
        loss.backward()
        opt.step()
    model.eval()
    with torch.no_grad():
        z = model.encode(x, edge_index, edge_weight)
        recon = model.decoder.forward_all(z).cpu().numpy()
    return (recon - recon.min()) / (recon.max() - recon.min() + 1e-12)


def evaluate(score_matrix, label, ground_truth, gene_names, num_genes):
    gt_matrix = ground_truth.loc[gene_names, gene_names].values.astype(int)
    triu = np.triu_indices(num_genes, k=1)
    y_true = gt_matrix[triu]
    y_score = score_matrix[triu]
    auroc = roc_auc_score(y_true, y_score)
    aupr = average_precision_score(y_true, y_score)
    best_f1, best_t = 0.0, 0.0
    for t in np.linspace(y_score.min(), y_score.max(), 50):
        pred = (y_score >= t).astype(int)
        f1 = f1_score(y_true, pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    print(f"[{label}] AUROC={auroc:.4f}  AUPR={aupr:.4f}  best-F1={best_f1:.4f} (thresh={best_t:.3f})")
    return {'label': label, 'auroc': auroc, 'aupr': aupr, 'best_f1': best_f1}

## 5. Run both conditions (3 iterations x 17 algorithms each, 150 epochs, embedding_dim=16)

In [ ]:
NUM_ITERATIONS = 3
EPOCHS = 150
EMBED_DIM = 16
x_tensor = torch.tensor(node_features, dtype=torch.float).to(DEVICE)

print("=== BASELINE: independent per-algorithm GCN-GAE, unweighted edges ===")
t0 = time.time()
agg_baseline = np.zeros((num_genes, num_genes))
n = 0
for it in range(NUM_ITERATIONS):
    for alg in algorithms:
        edge_index, edge_weight = alg_edges_baseline[alg]
        ew = edge_weight.to(DEVICE) if edge_weight is not None else None
        recon = train_gae(edge_index.to(DEVICE), ew, x_tensor, embedding_dim=EMBED_DIM, epochs=EPOCHS)
        agg_baseline += recon
        n += 1
    print(f"  iteration {it+1}/{NUM_ITERATIONS} done")
agg_baseline /= n
time_baseline = time.time() - t0
print(f"Baseline total time: {time_baseline:.1f}s")

In [ ]:
print("=== CORRELATION-WEIGHTED: same edges, weighted by |expression correlation| ===")
t0 = time.time()
agg_weighted = np.zeros((num_genes, num_genes))
n = 0
for it in range(NUM_ITERATIONS):
    for alg in algorithms:
        edge_index, edge_weight = alg_edges_weighted[alg]
        recon = train_gae(edge_index.to(DEVICE), edge_weight.to(DEVICE), x_tensor, embedding_dim=EMBED_DIM, epochs=EPOCHS)
        agg_weighted += recon
        n += 1
    print(f"  iteration {it+1}/{NUM_ITERATIONS} done")
agg_weighted /= n
time_weighted = time.time() - t0
print(f"Correlation-weighted total time: {time_weighted:.1f}s")

## 6. Results

In [ ]:
res_base = evaluate(agg_baseline, "Baseline (unweighted edges)", ground_truth, gene_names, num_genes)
res_corr = evaluate(agg_weighted, "Correlation-weighted edges", ground_truth, gene_names, num_genes)

print()
print(f"{'Metric':<12}{'Baseline':<15}{'Corr-weighted':<15}{'Winner'}")
for key in ['auroc', 'aupr', 'best_f1']:
    a, b = res_base[key], res_corr[key]
    winner = 'Baseline' if a > b else ('Corr-weighted' if b > a else 'tie')
    print(f"{key:<12}{a:<15.4f}{b:<15.4f}{winner}")

## 7. Conclusion

**Actual results from this run:**

| Metric | Baseline | Correlation-weighted |
|---|---|---|
| AUROC | 0.8071 | 0.8070 |
| AUPR | 0.1028 | 0.0960 |
| Best F1 | 0.1995 | 0.1882 |

**Correlation-weighting the existing edges did not help.** AUROC was essentially unchanged and AUPR/F1 were slightly worse than the unweighted baseline.

**Why this was worth testing anyway:** several of the 17 GRN algorithms already feeding this pipeline are themselves correlation/mutual-information based (CLR, PCIT, MRNET, MRNETB, ARACNE), so a naive "add correlation as an 18th vote" would have just double-counted signal those algorithms already contribute. Using correlation as an edge-*weight* modulator instead of a vote was a genuinely different, non-redundant way to bring expression data in -- it just didn't pay off with this simple implementation.

**Possible reasons it didn't help, if this is revisited later:**
- A hard 0.05 floor + `|correlation|` may be too blunt a confidence signal; a learned or rank-based weighting might behave differently.
- The GAE's reconstruction loss already rewards structural (topological) proximity; correlation strength doesn't necessarily align with which edges are structurally central to reconstruct.
- Only tested on DREAM4 network 1, 3 iterations, no hyperparameter search specific to this variant.

**Bottom line for now:** keep the existing unweighted-edge, identity-node-feature `FINAL_MODEL_GCN` approach as the baseline. Both "real expression as node features" and "correlation-weighted edges" were tested here and in the companion notebook, and neither beat it. The current final model remains the strongest validated approach so far.